# 04 結合の方式

**目的**: Spark が結合をどう実行するか（SortMergeJoin と BroadcastHashJoin）を見分け、どちらになるかを自分で制御できるようにする（70分）。

終わったときに説明できるようになること:

- SortMergeJoin と BroadcastHashJoin の違い。broadcast が速い理由と、使えない場面
- Spark が broadcast するかどうかを、何を見て決めているか
- AQE が実行中に結合方式を切り替える仕組み
- 偏ったキーで結合すると何が起きるか（発展）

データは `make data`（scale=1）で生成したものを前提にしている。

In [ ]:
from pyspark.sql import functions as F

from labkit import check_plan, get_spark, large, observe, partition_summary, plan_text

spark = get_spark("04_joins", {
    "spark.sql.adaptive.enabled": "false",
    "spark.sql.autoBroadcastJoinThreshold": "-1",   # 自動の broadcast を止める（実験3で戻す）
})

orders = large(spark, "orders")
customers = large(spark, "customers")
order_items = large(spark, "order_items")
products = large(spark, "products")


def summarize(joined):
    """結合結果を segment ごとに集計する。結合方式の違いだけを比べるための、共通の後処理。"""
    return joined.groupBy("segment").agg(F.count("*").alias("n_orders"), F.max("ordered_at").alias("last_ordered_at"))

---

## 実験1: SortMergeJoin

注文（500万行）に顧客（150万行）を結合する。

In [ ]:
# TODO 1-1: orders と customers を customer_id で結合する
smj = ...

In [ ]:
check_plan(smj, has=["SortMergeJoin"])
summarize(smj).explain()

with observe(spark, "SortMergeJoin"):
    rows = summarize(smj).collect()
print(*rows, sep="\n")

✏️ **観察: 計画の SortMergeJoin の下には、左右それぞれに何がある？ shuffle で書き出されたサイズは、orders 側と customers 側でそれぞれいくつ？**

（ここに書く）

SortMergeJoin は、(1) 両方のテーブルを結合キーで shuffle して、同じキーの行を同じパーティションに集め、(2) それぞれをキーで並べ替え、(3) 先頭から突き合わせる。どんなに大きいテーブル同士でも動くが、両側の全行が shuffle を通る。

---

## 実験2: broadcast ヒント

In [ ]:
# TODO 2-1: 同じ結合を、customers 側を broadcast するヒントを付けて書く
#   使うもの: F.broadcast(...)
bhj = ...

In [ ]:
check_plan(bhj, has=["BroadcastHashJoin", "BroadcastExchange"], lacks=["SortMergeJoin"])
summarize(bhj).explain()

with observe(spark, "BroadcastHashJoin"):
    rows = summarize(bhj).collect()
print(*rows, sep="\n")

✏️ **観察: 実験1と比べて、時間・ステージ数・shuffle の量はどう変わった？ 計画の orders 側に Exchange はある？**

（ここに書く）

BroadcastHashJoin は、小さいほうのテーブルをまるごとドライバに集めてハッシュ表にし、全タスクに配る。大きいほうのテーブルは、shuffle も並べ替えもせず、読みながらその場でハッシュ表を引く。

速いのは、大きいテーブルを動かさないから。その代わり、配るテーブルがメモリに載る大きさでなければならない。150万行の顧客テーブルは、このマシンでは配れたが、本番で「たまたま動いた」ヒントを残すと、データが増えた日にメモリ不足で落ちる。

---

## 実験3: Spark はいつ自動で broadcast するか

ヒントがなくても、Spark は「片側が十分小さい」と判断すれば broadcast する。判断の基準は `spark.sql.autoBroadcastJoinThreshold`（既定 10MB）で、**Spark が見積もったサイズ**がこれ以下なら broadcast する。

見積もりは `explain(mode="cost")` の `Statistics(sizeInBytes=...)` で見られる。

In [ ]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10 * 1024 * 1024)  # 既定値（10MB）に戻す


def estimated(df, label):
    """Spark が見積もった df のサイズ（計画のいちばん上の演算子の Statistics）を表示する。"""
    first = plan_text(df, "cost").split("== Optimized Logical Plan ==")[1].strip().splitlines()[0]
    print(f"{label:34s} {first.split('Statistics(')[1].rstrip(')')}")


small_area = customers.filter((F.col("prefecture") == "熊本県") & (F.col("segment") == "enterprise"))
print(f"small_area の実際の行数: {small_area.count():,} / customers 全体: {customers.count():,}")

In [ ]:
# TODO 3-1: 次の4つについて estimated(df, "ラベル") を呼び、見積もりサイズを表示する
#   (1) products
#   (2) customers
#   (3) small_area（熊本県の enterprise 顧客だけに絞ったもの。上のセルで定義済み）
#   (4) customers から customer_id の1列だけを select したもの
...

先に予想を書く。それぞれ結合したら、自動で broadcast される？（10MB = 10,485,760 バイト = 10.0 MiB と比べる）

| 結合 | 見積もり | 予想 | 結果 |
|---|---|---|---|
| order_items ⋈ products | | | |
| orders ⋈ customers | | | |
| orders ⋈ small_area | | | |
| orders ⋈ customers の customer_id だけ | | | |

In [ ]:
cases = {
    "order_items ⋈ products": order_items.join(products, "product_id"),
    "orders ⋈ customers": orders.join(customers, "customer_id"),
    "orders ⋈ small_area": orders.join(small_area, "customer_id"),
    "orders ⋈ customers の customer_id だけ": orders.join(customers.select("customer_id"), "customer_id"),
}
for name, df in cases.items():
    kind = "BroadcastHashJoin" if "BroadcastHashJoin" in plan_text(df) else "SortMergeJoin"
    print(f"{name:40s} {kind}")

✏️ **観察: 予想と違ったものは？ small_area は実際には数千行しかないのに、見積もりはどうなっていた？ 列を絞った (4) の見積もりは？**

（ここに書く）

統計情報を取っていないテーブルでは、Spark はファイルサイズから見積もる。列を絞れば見積もりは減るが、**filter でどれだけ行が減るかは分からない**ので、絞り込み後もサイズをほとんど減らさない。その結果、実際には数千行しかないテーブルなのに broadcast されない。

対処は3つある。

- 人間が分かっているなら `F.broadcast()` でヒントを付ける
- テーブルの統計を取っておく（`ANALYZE TABLE`）。統計を使った見積もり（コストベースの最適化）は、Databricks では既定で有効。素の Spark では `spark.sql.cbo.enabled` を有効にする必要がある
- AQE に任せる（次の実験）

---

## 実験4: AQE が実行中に結合方式を切り替える

In [ ]:
spark.conf.set("spark.sql.adaptive.enabled", "true")

adaptive = orders.join(small_area, "customer_id")   # 実験3で SortMergeJoin になった結合
print("=== 実行前 ===")
adaptive.explain()

with observe(spark, "AQE に任せる"):
    rows = adaptive.collect()
print(f"{len(rows):,} 行")

print("=== 実行後 ===")
adaptive.explain()

✏️ **観察: 実行前（Initial Plan）の結合方式は？ 実行後（Final Plan）は？ orders 側の shuffle は実行された？（shuffle書きのサイズを見る）**

（ここに書く）

AQE は、両側の shuffle が終わった時点で small_area 側の実際のサイズを見て、「これなら broadcast できる」と計画を書き換えた。見積もりが外れても、実行中に取り返してくれる。

ただし、切り替えを決めた時点で、orders 側の 500万行の shuffle はもう終わっている。最初から broadcast と分かっていた場合と比べてみる。

In [ ]:
# TODO 4-1: 同じ結合を、small_area に broadcast のヒントを付けて書く
hinted = ...
with observe(spark, "ヒントを付ける"):
    rows = hinted.collect()

✏️ **観察: 「AQE に任せる」と「ヒントを付ける」で、時間と shuffle の量はどう違った？**

（ここに書く）

AQE は、shuffle を実行してから直す。最初から小さいと分かっていれば、その shuffle 自体が要らない。ヒントや統計が今でも意味を持つのはこのためだ。

In [ ]:
spark.conf.set("spark.sql.adaptive.enabled", "false")

---

## 実験5（発展）: 偏ったキーで結合する

`02_partitions` で見たとおり、orders を customer_id で分けると、大口顧客の入ったパーティションだけが極端に大きくなる。SortMergeJoin はまさにこの分け方をするので、結合のステージで1つのタスクだけが長引く。

In [ ]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
partition_summary(orders.repartition(200, "customer_id"), "orders を customer_id で 200 分割")

with observe(spark, "偏ったキーの SortMergeJoin（AQE なし）"):
    rows = summarize(orders.join(customers, "customer_id")).collect()

Spark UI の Stages タブで、今のジョブのうち結合を実行したステージ（タスク数 200、shuffle 読みがあるもの）を開く。**Summary Metrics** の表で、`Duration` と `Shuffle Read Size / Records` の Median と Max を比べる。

✏️ **観察: 1タスクが処理した行数（Shuffle Read Records）の Max は、Median の何倍？**

（ここに書く）

AQE には、偏ったパーティションを見つけて分割する機能がある（skew join の最適化）。既定では「中央値の5倍より大きく、かつ 256MB より大きい」パーティションが対象になる。このデータは小さくて 256MB に届かないので、仕組みを見るためにしきい値を下げて試す。

In [ ]:
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", "1m")  # 既定は 256MB。実験用に下げる
spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", "1m")              # 既定は 64MB。実験用に下げる

skewed = summarize(orders.join(customers, "customer_id"))
with observe(spark, "偏ったキーの SortMergeJoin（AQE あり）"):
    rows = skewed.collect()

final_plan = plan_text(skewed)
print("skew の分割が", "働いた ✅" if "skew=true" in final_plan else "働かなかった（データ量やしきい値による）")
print("\n".join(line for line in final_plan.split("+- == Initial Plan ==")[0].splitlines()
                if "SortMergeJoin" in line or "AQEShuffleRead" in line))

✏️ **観察: Final Plan の SortMergeJoin と AQEShuffleRead に、何という印が付いた？**

（ここに書く）

偏りへの対処をまとめると:

| 方法 | 向いている場面 |
|---|---|
| broadcast する | 片側が小さい。shuffle 自体がなくなるので、偏りの影響を受けない |
| AQE の skew join に任せる | 両側が大きい。まず試す。しきい値は既定のままで良いことが多い |
| 偏ったキーだけ分けて処理する | 原因のキーが少数で、はっきりしている（NULL や「不明」を表す値など） |
| キーに乱数を足して分散させる（salting） | 上のどれも効かないときの最後の手段。コードが複雑になる |

---

## まとめ

✏️ **このノートブックで分かったことを、自分の言葉で3行にまとめる**

（ここに書く）

### 説明できるか確かめる

1. BroadcastHashJoin が SortMergeJoin より速いのはなぜ？ 使えないのはどんなとき？
2. Spark が自動で broadcast するかどうかは、何と何を比べて決まる？ その見積もりが外れやすいのはどんなとき？
3. `F.broadcast()` のヒントを本番のコードに書くときに、気をつけることは？
4. AQE が結合について実行中にやってくれることを2つ挙げられる？
5. 「ジョブの大半のタスクは数秒で終わるのに、1つだけ何分もかかる」。何が起きていて、何を確認する？

業務で動かしている Spark のジョブ（AWS Glue など）に、JDBC から読んだテーブル同士を結合している箇所があれば、`explain()` を出して、どの方式で結合されているかを見てみる。

In [ ]:
spark.stop()